The libraries such as OpenCV, Python csv module, MediaPipe, and Time

In [1]:
import cv2 as cv
import csv
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import time
import joblib 

The below code activates the camera and uses the MediaPipe landmarks

In [2]:
class Vision:
    def __init__(self):
        self.BaseOptions = mp.tasks.BaseOptions
        self.GestureRecognizer = mp.tasks.vision.GestureRecognizer
        self.GestureRecognizerOptions = mp.tasks.vision.GestureRecognizerOptions
        self.GestureRecognizerResult = mp.tasks.vision.GestureRecognizerResult
        self.VisionRunningMode = mp.tasks.vision.RunningMode

        self.options = self.GestureRecognizerOptions(
            base_options = self.BaseOptions(model_asset_path = 
                "C:/Users/jatep/OneDrive/Desktop/gesture_recognizer.task"),
            running_mode = self.VisionRunningMode.LIVE_STREAM,
            result_callback = self.print_result)

        self.gesture_key = {"0": "fist",
                            "1": "open_palm",
                            "2": "pointing",
                            "3": "thumbs_up",
                            "4": "peace_sign",
                            "5": "rock_on",
                            "6": "middle_finger",
                            "7": "ok_sign",
                            "8": "vulcan_salute",
                            "9": "shaka_sign"}
        
        self.cap = cv.VideoCapture(0)
        self.gesture_idx = None
        self.key_bool = False

    def camera(self) -> None:
        recognizer = self.GestureRecognizer.create_from_options(self.options)

        try:
            while True:
                ret, frame = self.cap.read()
                # Just like negation from discrete. If 'ret' is True then conditional makes
                # false, if 'ret' is False then conditional makes True.
                if not ret:
                    print("Frame not captured")
                
                new_frame = cv.flip(frame, 1)

                rgb_frame = cv.cvtColor(new_frame, cv.COLOR_BGR2RGB)
                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

                # MediaPipe time stamp in monotonic time (milliseconds)
                frame_timestamp_ms = int(time.monotonic() * 1000)
            
                recognizer.recognize_async(mp_image, frame_timestamp_ms)

                cv.imshow('Live Feed', new_frame)

                # Start recording
                key = cv.waitKey(1) & 0xFF

                # If a key is pressed update, else do nothing (no key press = 255('ÿ'))
                if key != 255:
                    check_key = chr(key)

                    if check_key == 'd':
                        print("Closing window")
                        break

                    elif check_key == ' ':
                        print("Stopped tracking")
                        self.key_bool = False

                    elif check_key in self.gesture_key.keys():
                        print(f"Started tracking {self.gesture_key[check_key]}")
                        self.gesture_idx = check_key
                        self.key_bool = True
                        
        finally:
            self.cap.release()
            recognizer.close()
            cv.destroyAllWindows()

    def print_result(self, result: GestureRecognizerResult, output_image: mp.Image, timestamp_ms: int) -> None:
        if result.hand_landmarks:
            if self.gesture_idx in self.gesture_key.keys():
                while self.key_bool:
                    gesture_name = self.gesture_key[self.gesture_idx]
                    
                    data = []

                    for land in result.hand_landmarks[0]:
                        data.append(land.x)
                        data.append(land.y)
                        data.append(land.z)

                    data.append(gesture_name)
                
                    with open('C:/Users/jatep/OneDrive/Desktop/gesture_data.csv.txt', 'a', newline = '') as csvfile:
                        writer = csv.writer(csvfile)
                        writer.writerow(data)
            
start_camera = Vision()
start_camera.camera() 

AttributeError: 'RandomForestClassifier' object has no attribute 'close'

Opens the 'gesture_data.csv' file to ensure the data is properly being saved

In [ ]:
with open('C:/Users/jatep/OneDrive/Desktop/gesture_data.csv.txt', 'r', newline = '') as csvfile:
    for row in csvfile:
        print(row)